Hasil eksperimen menunjukkan bahwa dari tiga algoritma klasifikasi yang diuji (ANN, LGBM, dan SVM) dengan tiga metode word embedding (Word2Vec, FastText, dan GloVe), kombinasi **SVM** dan **FastText** memberikan performa terbaik sehingga dipilih sebagai model yang digunakan pada tahap inferensi.

# Import Dependencies

## Libraries

In [ ]:
!pip install emoji Sastrawi gensim scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.4 MB/s eta 0:00:00


In [ ]:
import pickle
import re
import string

import emoji
import joblib
import nltk
import numpy as np
from nltk import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# custom modules
import slang_words
from utils import sentence_vector

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## Variables

In [ ]:
with open("stem_cache.pkl", "rb") as f:
  stem_cache = pickle.load(f)

with open("stopwords.pkl", "rb") as f:
  stopwords = pickle.load(f)

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

bundle = joblib.load("fasttex_svc.joblib")

embed_model = bundle["embed_model"]
scaler = bundle["scaler"]
encoder = bundle["encoder"]
model = bundle["model"]

## Functions

In [ ]:
def separate_emoji(text):
  """
  Separate emojis in a text by surrounding them with spaces.

  This function uses the `emoji.replace_emoji()` utility to detect emojis
  in a string and add spaces before and after each emoji. This is useful
  for text preprocessing tasks (e.g., NLP tokenization) so that emojis
  are treated as separate tokens instead of being attached to words.

  Parameters
  ----------
  text : str
    Input text that may contain emojis.

  Returns
  -------
  str
    The processed text where each emoji is separated by spaces.

  Raises
  ------
  ImportError
    If the `emoji` package is not installed.

  Notes
  -----
  This function requires the external library `emoji`. Install it with:

    pip install emoji

  Examples
  --------
  >>> separate_emoji("I love pizza🍕!")
  'I love pizza 🍕 !'
  """
  return emoji.replace_emoji(text, replace=lambda e, data: f" {e} ")

def get_stemmed_term(term: str):
  """
  Retrieve the stemmed form of a term using a cache dictionary.

  This function stems a given term using a stemmer and stores the result
  in a cache dictionary to avoid repeated stemming of the same term.
  If the term has already been processed, the cached result is returned
  directly, improving performance during large-scale text preprocessing.

  Parameters
  ----------
  term : str
    The word or token to be stemmed.

  Returns
  -------
  str
    The stemmed version of the input term.

  Notes
  -----
  This function assumes that two global variables exist:
  - `stemmer` : an object implementing a `.stem()` method.
  - `stem_cache` : a dictionary used to store previously stemmed terms.

  Examples
  --------
  >>> get_stemmed_term("berlari")
  'lari'

  >>> get_stemmed_term("memakan")  # retrieved from cache on repeated calls
  'makan'
  """
  if term not in stem_cache:
    stem_cache[term] = stemmer.stem(term)
  return stem_cache[term]


def inference(
    text: str,
    stopwords: set[str],
    stem_cache: dict[str, str],
    embed_model,
    scaler,
    encoder,
    model
    ) -> str:
    """
    Perform sentiment inference on a single text input using a trained model.

    This function applies the same preprocessing pipeline used during training,
    including text normalization, tokenization, stopword removal, stemming,
    and sentence embedding. The resulting feature vector is standardized using
    the provided scaler and passed to the trained model for prediction. The
    predicted label is then decoded using the label encoder.

    Parameters
    ----------
    text : str
        Input text to be classified.

    stopwords : Collection[str]
        A collection of stopwords used to filter tokens during preprocessing.

    stem_cache : dict[str, str]
        A dictionary used to cache stemming results. Keys are original tokens
        and values are their corresponding stemmed forms.

    embed_model : object
        Pretrained embedding model used to convert tokens into a sentence
        vector representation (e.g., Word2Vec, FastText, or similar).

    scaler : object
        A fitted feature scaler used to standardize the sentence vector
        before feeding it into the classifier.

    encoder : object
        A fitted label encoder used to convert encoded predictions back
        to their original label representation.

    model : object
        A trained classification model that implements a `predict()` method.
        The model may return either class labels or class probabilities.

    Returns
    -------
    str
        Predicted class label for the input text. If the preprocessing
        results in an empty token list after stopword removal, the function
        returns `"neutral"`.

    Notes
    -----
    This function assumes the following helper utilities exist:
    - `separate_emoji()` for spacing emojis in text
    - `slang_words.fix_slangwords()` for slang normalization
    - `get_stemmed_term()` for cached stemming
    - `sentence_vector()` for generating sentence embeddings
    - `word_tokenize()` for tokenization

    Examples
    --------
    >>> inference("Filmnya bagus banget!", stopwords, stem_cache, embed_model, scaler, encoder, model)
    'positive'
    """

    text = text.lower()

    text = separate_emoji(text)

    text = re.sub(r"\d+", "", text)
    text = re.sub(f"[{string.punctuation}]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    text = slang_words.fix_slangwords(text)

    text = word_tokenize(text)

    text = [t for t in text if t not in stopwords]

    if len(text) == 0:
        return "neutral"

    text = [get_stemmed_term(t) for t in text]

    text = np.array([sentence_vector(text, embed_model)])

    text = scaler.transform(text)

    pred = model.predict(text)

    if np.ndim(pred) > 1:
        pred = np.argmax(pred, axis=1)

    return encoder.inverse_transform(pred)[0]

# Inference

Berikut adalah contoh ulasan aplikasi **BSI Mobile** yang ditemukan di **Play Store**:

"""<br>
Hallo Kak. . . Saya Nasabah Baru yang menggunakan Aplikasi dan Telah Membuka Rekening Easy Wadiah Via Online,. Mohon BANTUAN. . . Saya Inggin Login untuk Transaksi Rekening Tapi Aplikasi BYOND,. Mengatakan Kalau Sandi Salah,Padahal Saya Sudah memasukan Sandi Dengan Benar,. Byond meminta saya input nomer kartu debit. Tapi Debit Saya Belum dikirim. dan Aplikasi byon Sering memberitahukan untuk update . Mohon Bantuannya. . Karna Saya Nasabah Baru ,. dan Inggin Menjadi Nasabah Tetap BSI
<br>"""

Selanjutnya akan dilakukan **inference** untuk mengklasifikasikan sentimen dari ulasan tersebut menggunakan model yang telah dilatih.

In [24]:
text = input("Masukkan teks: ")

print(f"""
Review tersebut memiliki sentimen:
{inference(text, stopwords, stem_cache, embed_model, scaler, encoder, model)}
""")

Masukkan teks: Hallo Kak. . . Saya Nasabah Baru yang menggunakan Aplikasi dan Telah Membuka Rekening Easy Wadiah Via Online,. Mohon BANTUAN. . . Saya Inggin Login untuk Transaksi Rekening Tapi Aplikasi BYOND,. Mengatakan Kalau Sandi Salah,Padahal Saya Sudah memasukan Sandi Dengan Benar,. Byond meminta saya input nomer kartu debit. Tapi Debit Saya Belum dikirim. dan Aplikasi byon Sering memberitahukan untuk update . Mohon Bantuannya. . Karna Saya Nasabah Baru ,. dan Inggin Menjadi Nasabah Tetap BSI

Review tersebut memiliki sentimen:
neutral

